# SBOM-to-Audit — Stage 2.0.1 Conflict-Lifecycle Checkpoint

This short clean-room checkpoint verifies the corrective conflict-lifecycle build from an exact GitHub checkout in an isolated Python environment. It does not edit or push repository files.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with the corrective tag after it is created.
print("Repository:", REPO_URL)
print("Reference:", REF)

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV = Path("/content/sbom_to_audit_stage201_venv")
for path in (WORKDIR, VENV):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    check=True,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")

In [ ]:
import json

RELEASE_REPORT = Path("/content/stage2_0_1_release_validation.json")
subprocess.run(
    [str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    check=True,
)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release.get("errors")
assert len(release["deterministic_hashes"]) == 6
print("PASS: canonical release gate")
print(json.dumps(release["deterministic_hashes"], indent=2))

In [ ]:
OUTPUT_ROOT = Path("/content/stage2_0_1_outputs")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "sbom_to_audit.cli",
        "--scenario",
        "data/scenarios/ghost_logger.yaml",
        "--output-root",
        str(OUTPUT_ROOT),
    ],
    check=True,
)
conflict_report = json.loads((OUTPUT_ROOT / "conflict_reports/ghost_logger.json").read_text())
pack = json.loads((OUTPUT_ROOT / "evidence_packs/ghost_logger.json").read_text())
assert pack["schema_version"] == "0.2"
assert pack["orchestration_metrics"]["C_t"] is False
assert conflict_report["detected_conflicts"] == 1
assert conflict_report["active_conflicts"] == 0
assert conflict_report["resolved_conflicts"] == 1
conflict = conflict_report["conflicts"][0]
assert conflict["status"] == "resolved"
assert conflict["detected_at_event_id"] == "EVT-GL-010H"
assert conflict["resolved_at_event_id"] == "EVT-GL-014H"
assert conflict["resolution_artifact_ids"] == ["ART-DEC-RESOLVE-001"]
assert [item["status"] for item in conflict["lifecycle"]] == ["active", "resolved"]
audit_text = (OUTPUT_ROOT / "audit_ledgers/ghost_logger.jsonl").read_text()
assert "evidence_conflict_detected" in audit_text
assert "evidence_conflict_resolved" in audit_text
print("PASS: intentional conflict detected at T+10h")
print("PASS: conflict explicitly resolved at T+14h")
print("PASS: final C_t agrees with zero active conflict records")

In [ ]:
import datetime as dt
import hashlib
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "release_status": release["status"],
}
ENV_PATH = Path("/content/stage2_0_1_colab_environment.json")
ENV_PATH.write_text(json.dumps(environment, indent=2) + "\n")
BUNDLE = Path("/content/stage2_0_1_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV_PATH, ENV_PATH.name)
    for rel in (
        "conflict_reports/ghost_logger.json",
        "audit_ledgers/ghost_logger.jsonl",
        "state_logs/ghost_logger.csv",
        "metrics/ghost_logger_metrics.json",
    ):
        archive.write(OUTPUT_ROOT / rel, f"outputs/{rel}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("SHA-256:", digest)
print("Download the bundle from Colab Files.")